*(Oh boy `evcxr` does **not** like the macro hacks I'm doing)*

Dependencies:

In [2]:
:dep rucompart = {path = "."}
:dep tokio = {version = "1.34.0", features = ["full"]}
:dep futures = "0.3.31"
:dep tarpc = { version = "0.37.0", features = ["serde-transport", "serde-transport-bincode", "unix", "tcp"] }
:dep libc = "0.2.181"

Declare a compartment:

In [3]:
// These errors are probably due to something evcxr is doing to the code?
// They don't occur in normal use.
#![allow(unused_imports, unused_braces)]
#[macro_use]
extern crate tarpc;
#[macro_use]
extern crate rucompart;

pub mod some_compartment {
	/// API surface
	#[service]
	pub trait SomeCompartment {
		async fn hello(name: String) -> String;
	
		async fn sum(a: i64, b: i64) -> i64;
	
		async fn sort(list: Vec<i64>) -> Vec<i64>;
	}
	
	/// Marker zero-size-type for a particular implementation of the service
	#[derive(Clone)]
	pub struct SomeCompartmentService;
	
	/// The actual implementation
	impl SomeCompartment for SomeCompartmentService {
		async fn hello(self, _context: ::tarpc::context::Context, name: String) -> String {
			format!("Hello {name}! The compartment's pid is {}", unsafe {
				libc::getpid()
			})
		}
	
		async fn sum(self, _context: ::tarpc::context::Context, a: i64, b: i64) -> i64 {
			a + b
		}
	
		async fn sort(self, _context: ::tarpc::context::Context, mut list: Vec<i64>) -> Vec<i64> {
			list.sort();
			list
		}
	}
	
	// A macro to do some annoying boilerplate:
	// In theory, it should be possible to do this with generics,
	// but due to some intricacies of the type system implementation, it is not.
	compartmentalize!(
		"SOME_COMPARTMENT",
		ServeSomeCompartment,
		SomeCompartmentService,
		SomeCompartmentClient,
		async fn setup(&mut self, mode: rucompart::CompartmentMode) -> Result<_, std::io::Error> {Ok(())}
	);
}

(ALL fork-compartments must be set up before this next part. This is enforced at runtime.)

In [4]:
#[macro_use]
extern crate rucompart;
use some_compartment::*;
use tokio::runtime::Runtime;

// We can't spawn separate processes easily in a Jupyter notebook
// without moving their code to another file, so I'll be using the
// fork backend here.

// If we passed Some(String) here, this macro would parse a TCP or
// Unix socket address from that string and connect to it.
// Because we pass `None`, it will use the fork backend.
let compartment_client = compartment_connect!(None, SomeCompartmentService, SomeCompartmentClient);
Runtime::new().unwrap().block_on(async move {
    // The construction of the client has some steps that require an
    // async runtime, so they have to be delayed until we start it.
    let compartment_client = compartment_client.await;
    println!("A");
});

Error: remove the whole `use` item